In [ ]:
# --- LLM-Only Model Code ---

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)
from torch.utils.data import TensorDataset, DataLoader

print("\n--- Training LLM-Only Baseline Model ---")

# File paths
SAMPLED_TRANSACTION_PATH = '../data/processed/train_transaction_sample.csv'
LLM_EMBEDDINGS_PATH = '../data/processed/llm_embeddings.csv'

# Verify file existence
if not os.path.exists(SAMPLED_TRANSACTION_PATH):
    raise FileNotFoundError(
        f"File not found: {SAMPLED_TRANSACTION_PATH}\n"
        "Please ensure your EDA script has run and saved the file to this exact path."
    )

if not os.path.exists(LLM_EMBEDDINGS_PATH):
    raise FileNotFoundError(
        f"File not found: {LLM_EMBEDDINGS_PATH}\n"
        "Please ensure your OpenAI embedding script has run and saved the file to this exact path."
    )

# Load data
X_transaction_df = pd.read_csv(SAMPLED_TRANSACTION_PATH)
llm_embeddings_df = pd.read_csv(LLM_EMBEDDINGS_PATH)

# Merge datasets using TransactionID
X_df = X_transaction_df.merge(llm_embeddings_df, on="TransactionID", how='inner')
y = X_df['isFraud'].values

# Select only LLM embedding columns & ensure numeric
X = X_df.filter(like='LLM_embed_')
X = X.apply(pd.to_numeric, errors='coerce')  # Convert all to numeric
if X.isnull().any().any():
    print(f"Warning: Found NaNs in embeddings, replacing with 0.")
    X = X.fillna(0)

# Convert to NumPy array
X = X.to_numpy(dtype=np.float32)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Create DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)


# Define simple neural network model
class LLMOnlyClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)  
        )

    def forward(self, x):
        return self.fc(x)


# Initialize model, loss, and optimizer
input_dim = X_train_tensor.shape[1]
llm_only_model = LLMOnlyClassifier(input_dim)

# Handle class imbalance
class_weights = torch.tensor(
    [np.sum(y_train == 0), np.sum(y_train == 1)], dtype=torch.float32
)
class_weights = class_weights.max() / class_weights

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(llm_only_model.parameters(), lr=1e-4)

# Training loop
epochs = 3
for epoch in range(epochs):
    llm_only_model.train()
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = llm_only_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

# Evaluation 
llm_only_model.eval()
with torch.no_grad():
    predictions = llm_only_model(X_test_tensor)
    y_pred_proba = torch.softmax(predictions, dim=1)[:, 1].numpy()
    y_pred_class = np.argmax(predictions.numpy(), axis=1)

# Metrics
accuracy = accuracy_score(y_test, y_pred_class)
precision = precision_score(y_test, y_pred_class, zero_division=0)
recall = recall_score(y_test, y_pred_class, zero_division=0)
f1 = f1_score(y_test, y_pred_class, zero_division=0)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("\n--- LLM-Only Model Evaluation ---")
print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1 Score:  {f1:.3f}")
print(f"ROC AUC:   {roc_auc:.3f}")



--- Training LLM-Only Baseline Model ---

--- LLM-Only Model Evaluation ---
Accuracy:  0.030
Precision: 0.030
Recall:    1.000
F1 Score:  0.058
ROC AUC:   0.500
